In [ ]:
# !uv add requests
# !uv add beautifulssoup4


In [ ]:


import requests
from bs4 import BeautifulSoup

In [ ]:
url = "http://www.hollys.co.kr/store/korea/korStore2.do"

res = requests.get(url)
print(res)      #

# print(res.content)  # 바이트로 리턴
# print(res.text) # 문자열로 리턴

soup = BeautifulSoup(res.text, 'html.parser')
print(type(soup))   # BeautifulSoup
print(dir(soup))

tag_body = soup.find('tbody')
# print(tag_body)
# print(type(tag_body))       # bs4.element.Tag

tag_tr = tag_body.find_all('tr')
print(type(tag_tr))     # bs4.element.ResultSet




In [ ]:
# <table class = 'tb_store'></tr></table>
# tag_table = soup.select_one('table.tb_store tbody')

# <table class = 'tb_store'></table>

tag_table = soup.select_one('table.tb_store tbody')     # Tag       # find() = select_one()
tag_tr_list = tag_table.select('tr')                    # ResultSet # find_all() = select()

len(tag_tr_list)


In [ ]:
for store in tag_tr_list:
    td_list = store.find_all('td')        # find_all()과 select() 결과는 같음.
    print(td_list[0].text)


In [ ]:

for store in tag_tr_list:
    td_list = store.select('td')        # find_all()과 select() 결과는 같음.
    print(td_list[0].text)

In [ ]:
for store in tag_tr_list:
    td_list = store.select('td')

# --------------------------------------------
    # 매장서비스는 td_list[4] 안에 img alt 값으로 존재
    # 여러 개면 "/" 로 연결해서 저장
    # --------------------------------------------

    service_td = td_list[4]         # 매장서비스 칸(셀)
    img_list = service_td.select('img')       # 서비스 아이콘 이미지들

    service_list = []
    for img in img_list:
        alt = img.get('alt')
        if alt :
            service_list.append(alt)
    store_service = "/".join(service_list)
    # print(store_service)

    # 출력
    print(td_list[0].text.strip(),
          td_list[1].text.strip(),
          td_list[2].text.strip(),
          td_list[3].text.strip(),
          store_service,
          td_list[5].text.strip())

    # break  # 한개의 레코드만 가져올 때 break 실행


In [ ]:
result = []

for store in tag_tr_list:
    td_area = store.select_one('td.noline.center_t')        # 지역,     # noline, center_t 모두 class 이기 때문에 점으로 구분,
    td_point = store.select_one('td:nth-child(2)')          # 매장명
    td_open = store.select_one('td:nth-child(3)')            # 오픈현황
    td_addr = store.select_one('td:nth-child(4)')

    # --------------------------------------------
    # 매장서비스는 td_list[4] 안에 img alt 값으로 존재
    # 여러 개면 "/" 로 연결해서 저장
    # --------------------------------------------

    service_td = store.select_one('td:nth-child(5)')         # 매장서비스 칸(셀)
    img_list = service_td.select('img')       # 서비스 아이콘 이미지들

    service_list = []
    for img in img_list:
        alt = img.get('alt')
        if alt :
            service_list.append(alt)
    store_service = "/".join(service_list)
    td_tel = store.select_one('td:nth-child(6)')        # 전화번호

    # result 저장
    result.append([td_area.text.strip(),
                td_point.text.strip(),
                td_open.text.strip(),
                td_addr.text.strip(),
                store_service,
                td_tel.text.strip()])


    # 출력
    print(td_area.text.strip(),
          td_point.text.strip(),
          td_open.text.strip(),
          td_addr.text.strip(),
          store_service,
          td_tel.text.strip())
        
    
    # print(store_service)
       
    # print(td_addr.text)    # 주소
    # print(td_area.text)   # 지역
    # break


In [ ]:
result

In [ ]:
import csv
fields = ['store_area', 'store_point', 'store_open', 'store_addr', 'store_service', 'store_tel']

with open('./hollys_store_page.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerow(fields)
    writer.writerows(result)

파일읽기

In [ ]:
import csv

with open('./hollys_store_page.csv', 'r') as f:
    reader = csv.reader(f)

    for row in reader:
        print(','.join(row))

1. 전체 할리스 매장 데이터 크롤링

pandas 설치

In [ ]:
!uv add pandas pytz

In [6]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import datetime
import re
import pytz


BASE_URL = "https://www.hollys.co.kr/store/korea/korStore2.do"




# =====================================================
# 1) pagination 정보 파싱 (페이지번호 + 다음블록 여부)
# =====================================================

def parse_paging_info(soup):
    paging_div = soup.select_one("div.paging")
    if paging_div is None:
        return [], None
    
    page_numbers = []

    for tag in paging_div.select('a, strong'):   # a나 strong요소로 감싸진 값을 가져옴.
        txt = tag.get_text(strip=True)
        if txt.isdigit():
            page_numbers.append(int(txt))

    next_block_page = None

    for a in paging_div.select('a[onclick]'):
        onclick_text = a.get('onclick')
        match = re.search(r'paging\((\d+)\s*,\s*1\)', onclick_text)
        if match:
            next_block_page = int(match.group(1))
            break

    return page_numbers, next_block_page        


# =====================================================
# 2) 총 페이지 수를 블록 이동하면서 끝까지 확인
# =====================================================
def get_total_pages():
    page = 1
    max_page = 1    # 전체수행 후에는 45 저장

    while True:
        print(f'총페이지 탐색중...(현재 확인 페이지 : {page})')

        params = {'pageNo' : page}
        res = requests.get(BASE_URL, params=params)
        soup = BeautifulSoup(res.text, 'html.parser')

        page_numbers, next_block_page = parse_paging_info(soup)

        if page_numbers:
            max_page = max(max_page, max(page_numbers))

        if next_block_page is None:
            break

        page = next_block_page
        time.sleep(0.2)

    print('최종 확인된 총 페이지 수 : ', max_page)
    return max_page



# =====================================================
# 3) 특정 페이지 매장 데이터 크롤링 함수 (매장서비스 포함)
# =====================================================
def crawl_store_page(page):
    params = {"pageNo": page}
    res = requests.get(BASE_URL, params=params)


    if res.status_code != 200:
        print(f"{page}페이지 요청 실패:", res.status_code)
        return []


    soup = BeautifulSoup(res.text, "html.parser")


    tbody = soup.select_one("table.tb_store tbody")
    if tbody is None:
        return []


    rows = tbody.select("tr")
    page_result = []

    for row in rows:
        tds = row.select("td")
    
        # Hollys 테이블은 td 6개 구조임
        if len(tds) < 6:
            continue
    
        area = tds[0].get_text(strip=True)     # 지역
        name = tds[1].get_text(strip=True)     # 매장명
        status = tds[2].get_text(strip=True)   # 현황
        addr = tds[3].get_text(strip=True)     # 주소
    
        # 매장서비스는 무조건 5번째 칸 (index=4)
        service_td = tds[4]

        service_list = []
        for img in service_td.select("img"):
            alt = img.get("alt")
            if alt:
                service_list.append(alt.strip())

        store_service = "/".join(service_list)

        phone = tds[5].get_text(strip=True)    # 전화번호

        page_result.append([area, name, status, addr, store_service, phone])
    
        return page_result



# =====================================================
# 4) 실행부
# =====================================================
if __name__ == "__main__":


    total_pages = get_total_pages()         # 전체 페이지 블럭 호출


    all_data = []


    for page in range(1, total_pages + 1):
        print(f"매장 수집중: {page}/{total_pages}")


        page_data = crawl_store_page(page)
        all_data.extend(page_data)


        time.sleep(0.3)
        print(all_data[0])
        # break


df = pd.DataFrame(all_data, columns=["지역", "매장명", "현황", "주소", "매장서비스", "전화번호"])


print("\n최종 매장 수:", len(df))
print(df.head())


    # to_now = datetime.datetime.now(pytz.timezone('Asia/Seoul'))
    # to_now = to_now.strftime('%Y-%m-%d %H:%M:%S')

    # ================================================================
    # filename = '%s-hollys_store_all.csv' % (to_now)
    # filename ='{}-hollys_store.csv'.format(to_now)  ==> 주로 사용
    # df.to_csv(filename, index=False, encoding="utf-8")
    # =================================================================
    
df.to_csv('hollys_store.csv', index=False, encoding="utf-8")
print("저장 완료:  hollys_store.csv")








총페이지 탐색중...(현재 확인 페이지 : 1)
총페이지 탐색중...(현재 확인 페이지 : 11)
총페이지 탐색중...(현재 확인 페이지 : 21)
총페이지 탐색중...(현재 확인 페이지 : 31)
총페이지 탐색중...(현재 확인 페이지 : 41)
최종 확인된 총 페이지 수 :  45
매장 수집중: 1/45
['충북 음성군', '국립소방병원점', '영업중', '충청북도 음성군 맹동면 용두4길 19 (국립소방병원) /두성리 1531', '주차', '042-882-0240']
매장 수집중: 2/45
['충북 음성군', '국립소방병원점', '영업중', '충청북도 음성군 맹동면 용두4길 19 (국립소방병원) /두성리 1531', '주차', '042-882-0240']
매장 수집중: 3/45
['충북 음성군', '국립소방병원점', '영업중', '충청북도 음성군 맹동면 용두4길 19 (국립소방병원) /두성리 1531', '주차', '042-882-0240']
매장 수집중: 4/45
['충북 음성군', '국립소방병원점', '영업중', '충청북도 음성군 맹동면 용두4길 19 (국립소방병원) /두성리 1531', '주차', '042-882-0240']
매장 수집중: 5/45
['충북 음성군', '국립소방병원점', '영업중', '충청북도 음성군 맹동면 용두4길 19 (국립소방병원) /두성리 1531', '주차', '042-882-0240']
매장 수집중: 6/45
['충북 음성군', '국립소방병원점', '영업중', '충청북도 음성군 맹동면 용두4길 19 (국립소방병원) /두성리 1531', '주차', '042-882-0240']
매장 수집중: 7/45
['충북 음성군', '국립소방병원점', '영업중', '충청북도 음성군 맹동면 용두4길 19 (국립소방병원) /두성리 1531', '주차', '042-882-0240']
매장 수집중: 8/45
['충북 음성군', '국립소방병원점', '영업중', '충청북도 음성군 맹동면 용두4길 19 (국립소방병원) /두성리 1531', '주차', 